In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = '8.5' AND Value = '100'
      - primary_product_serv matches the predefined list of 14 categories. (Converted 
        to UPPER() for case-insensitive matching).
   3. COMMA EXCEPTION HANDLING: Protects specific unit names containing commas (e.g., 
      'TECE - Strategy, Change...') using the '[COMMA]' placeholder swap to prevent 
      improper splitting during the explosion phase.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists. Restores '[COMMA]' to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax.
   6. DUAL-BRIDGE JOIN: Matches on either the String Name OR the Numeric BusinessID 
      provided in the mapping table's Assessable_Unit_ID column.
   7. FINAL OUTPUT: Binary 'Yes' / 'No' output based on whether any matched high-risk 
      engagements exist for the AU.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings (Accommodates mixed ID/Name format)
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract columns and apply Comma Protection Dictionary
    SELECT 
        EngagementNumber,
        KPI_Number,
        Value,
        primary_product_serv,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    -- 4. Filter based on KPI 8.5 OR target marketing/printing categories
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE 
        (TRIM(KPI_Number) = '8.5' AND TRIM(Value) = '100')
        OR 
        UPPER(TRIM(primary_product_serv)) IN (
            'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
            'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
            'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
            'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
            'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
        )
),

Flattened_LOBs AS (
    -- 5. Explode comma-separated strings into individual rows and restore commas
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 6. Map expanded rows to AU using Dual-Bridge and count matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
    -- DUAL-BRIDGE JOIN: Supports mapping table's mix of Numeric ID or String Name
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 7. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID, 
    a.AU_Name, 
    a.Business_Segment,
    'TP03' AS QuestionID,               
    
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - Traceability Verification
   PURPOSE: Isolates Service Category Flagging, Comma Exception tracing, and 
   the Dual-Bridge Master AU lookup status for troubleshooting.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Filtered_TP_Data AS (
    SELECT 
        p.EngagementNumber,
        p.KPI_Number,
        p.Value,
        p.primary_product_serv,
        p.owning_lob AS Raw_Col_K,
        
        -- Exception Protection Trace
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    WHERE p.EngagementNumber IS NOT NULL
      AND (
          (TRIM(p.KPI_Number) = '8.5' AND TRIM(p.Value) = '100') 
          OR 
          UPPER(TRIM(p.primary_product_serv)) IN (
              'MARKETING MEDIA SERVICES', 'MARKETING AND DISTRIBUTION', 'DIRECT MARKETING PRINT SERVICE',
              'PRINT ADVERTISING', 'PRINTING', 'MANAGEMENT AND BUSINESS PROFESSIONALS AND ADMINISTRATIVE SERVICES',
              'MARKETING ANALYSIS', 'PUBLICITY AND MARKETING SUPPORT SERVICES', 'MARKETING COMMUNICATIONS AGENCIES',
              'ADVERTISING AGENCY SERVICES', 'BUSINESS FORMS OR QUESTIONNAIRES', 'SPECIALIZED WAREHOUSING AND STORAGE',
              'STATIONERY OR BUSINESS FORM PRINTING', 'PUBLISHED PRODUCTS'
          )
      )
),

Flattened AS (
    -- Safely explode using the protected strings!
    SELECT 
        f.EngagementNumber,
        f.KPI_Number,
        f.Value,
        f.primary_product_serv,
        f.Raw_Col_K,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Restored_Chunk
    FROM Filtered_TP_Data f
    LATERAL VIEW explode(split(concat_ws(',', f.safe_owning_lob, f.safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    f.KPI_Number,
    f.Value,
    f.primary_product_serv,
    f.Raw_Col_K AS Original_LOB_String,
    f.Restored_Chunk AS Individual_Mapped_LOB,
    map.TPRM_Units AS Matched_In_Mapping_Table,
    map.Assessable_Unit_ID AS Mapping_Table_Bridge_Value,
    mast.Master_Numeric_ID AS Successfully_Bridged_To_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Bridge_Status
FROM Flattened f
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Restored_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
    OR TRIM(map.Assessable_Unit_ID) = TRIM(mast.Master_Numeric_ID)
ORDER BY f.EngagementNumber;